In [ ]:
"""
Artificial Neural Network (ANN) for Predicting Compressive Strength
Author: Haseeb Ahmad
Description:
This script trains a feedforward ANN model to predict compressive strength
of gypsum-based composites using mix design parameters.
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


# ==============================
# Configuration
# ==============================

DATA_PATH = "../data/gypsum_data.csv"
FIGURE_PATH = "../figures"
RANDOM_STATE = 42
TEST_SIZE = 0.2
EPOCHS = 500
BATCH_SIZE = 16

os.makedirs(FIGURE_PATH, exist_ok=True)


# ==============================
# Load Dataset
# ==============================

def load_data(path):
    df = pd.read_csv(path, header=2)

    X = df.iloc[0:161, [4, 5, 6, 7, 8, 9, 10]].astype(float).values
    y = df.iloc[0:161, 13].astype(float).values

    return X, y


# ==============================
# Build ANN Model
# ==============================

def create_model(input_dim):
    model = keras.Sequential([
        layers.Dense(64, activation='relu', input_shape=(input_dim,)),
        layers.Dropout(0.2),
        layers.Dense(32, activation='relu'),
        layers.Dense(16, activation='relu'),
        layers.Dense(1)
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae']
    )

    return model


# ==============================
# Evaluation
# ==============================

def evaluate_model(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)

    print("\nModel Performance on Test Set")
    print("--------------------------------")
    print(f"MSE  : {mse:.4f}")
    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R²   : {r2:.4f}")

    return mse, mae, rmse, r2


# ==============================
# Plot Functions
# ==============================

def plot_training_history(history):
    plt.figure(figsize=(8, 4))
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURE_PATH, "training_history.png"), dpi=300)
    plt.show()


def plot_actual_vs_predicted(y_true, y_pred):
    x_vals = np.linspace(y_true.min(), y_true.max(), 200)

    plt.figure(figsize=(5, 4))
    plt.scatter(y_true, y_pred, alpha=0.7, edgecolors='k')
    plt.plot(x_vals, x_vals, 'r--', label="Perfect Fit")

    plt.xlabel("Actual Compressive Strength (MPa)")
    plt.ylabel("Predicted Compressive Strength (MPa)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURE_PATH, "actual_vs_predicted.png"), dpi=300)
    plt.show()


def plot_feature_importance(model, X_test, y_test, feature_names):
    result = permutation_importance(
        model,
        X_test,
        y_test,
        scoring='neg_mean_squared_error',
        n_repeats=10,
        random_state=RANDOM_STATE
    )

    importances = np.abs(result.importances_mean)
    indices = np.argsort(importances)

    plt.figure(figsize=(6, 4))
    plt.barh(range(len(importances)), importances[indices])
    plt.yticks(range(len(importances)), np.array(feature_names)[indices])
    plt.xlabel("Permutation Importance")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURE_PATH, "feature_importance.png"), dpi=300)
    plt.show()


# ==============================
# Main Execution
# ==============================

def main():

    # Load data
    X, y = load_data(DATA_PATH)

    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE
    )

    # Scale
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train_scaled = scaler_X.fit_transform(X_train)
    X_test_scaled = scaler_X.transform(X_test)

    y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1))

    # Build model
    model = create_model(X_train_scaled.shape[1])
    model.summary()

    # Early stopping
    early_stopping = keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=50,
        restore_best_weights=True
    )

    # Train
    history = model.fit(
        X_train_scaled,
        y_train_scaled,
        validation_split=0.2,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[early_stopping],
        verbose=1
    )

    # Predict
    y_pred_scaled = model.predict(X_test_scaled)
    y_pred = scaler_y.inverse_transform(y_pred_scaled)

    # Evaluate
    evaluate_model(y_test, y_pred)

    # Plots
    plot_training_history(history)
    plot_actual_vs_predicted(y_test, y_pred)

    feature_names = [
        "Gypsum Strength",
        "Gypsum Quantity",
        "Water Quantity",
        "Water/Gypsum Ratio",
        "Wheat Straw",
        "CaCl2",
        "Ca(OH)2"
    ]

    plot_feature_importance(model, X_test_scaled, y_test, feature_names)


if __name__ == "__main__":
    main()